# E2 — QSVC vs SVC clásica (protocolo de Kakavand + ajuste de escala `s`)

Replica la afirmación del informe: la SVM con kernel cuántico (QSVC) es ejecutable hoy, pero en
estudios con validación estadística no supera a la SVM clásica en datos tabulares
(Kakavand et al. 2026), y su desempeño depende del ancho de banda / escala de las
características (Shaydulin & Wild 2022; Flórez et al. 2025).

**Qué hace este notebook:** evalúa con validación cruzada anidada (5 externos × 3 internos,
estratificada) una SVM con kernel cuántico de fidelidad (`zzfm`, `rot2dof`; statevector exacto)
y SVM clásicas (`rbf`, `linear`) en `breast_cancer`, `banknote` (submuestra de 300) y `haberman`.
El preprocesamiento (estandarización + PCA) se ajusta **solo con el entrenamiento de cada
pliegue externo**. Guarda una fila por `(dataset, kernel, mode, seed, outer_fold)` en
`results/qsvc/<timestamp>/runs.csv`.

**Dos modos:**
- `kakavand`: reproduce el protocolo original (solo se ajusta `C`; escala `s = 1` fija).
- `scale_tuned`: **extensión propia, no es parte del protocolo de Kakavand.** La escala de las
  características `s` (`x → s·x` tras el preprocesamiento) se ajusta en la CV interna junto con
  `C`. La motivación es la dependencia del kernel cuántico con su ancho de banda
  (Shaydulin & Wild 2022; Flórez et al. 2025); Kakavand solo la menciona como trabajo futuro
  ("bandwidth-aware encoding", Sec. 7).

Se reutiliza el código de Kakavand (`qsvm_benchmark`, CC BY 4.0) para datasets,
preprocesamiento, mapas de características y kernels; **no se modifica**, solo se importa. No se
usa su `run_nested_cv` (calcula el kernel con preprocesamiento global y solo ajusta `C`); el
bucle propio sigue a `run_nested_cv_leakage_free`.

Todo es simulación. No se generan gráficos, tablas resumen ni conclusiones.

Secciones: 1 Configuración · 2 Utilidades · 3 Datos y preprocesamiento · 4 Kernels ·
5 CV anidada · 6 Pruebas · 7 Barrido · 8 Vista previa.

## 1. Configuración

In [ ]:
import csv, hashlib, itertools, json, logging, platform, subprocess, sys, tempfile, time, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import SVC

from qiskit.quantum_info import Statevector

# Código de Kakavand et al. (QSVM-Benchmark-Paper, CC BY 4.0): solo se importa, no se modifica
from qsvm_benchmark.backends.simulators import NoiseConfig, create_noisy_backend
from qsvm_benchmark.datasets import DatasetRegistry
from qsvm_benchmark.feature_maps import build_feature_map
from qsvm_benchmark.kernels import KernelCache, LinearKernel, compute_kernel_hash
from qsvm_benchmark.kernels import compute_noisy_density_matrix_kernel, compute_statevector_kernel
from qsvm_benchmark.kernels.classical import gamma_scale
from qsvm_benchmark.preprocess import build_preprocessor

# ----------------------------- Parámetros de ejecución -----------------------------
QUICK = True          # True: config rápida (1 semilla). False: barrido completo (5 semillas).
RESUME = False        # True: retoma OUTPUT_DIR saltando las corridas ya guardadas.
OUTPUT_DIR = None     # None -> results/qsvc/<timestamp>. Con RESUME=True, indicar la carpeta a retomar.
NOISE = False         # True: agrega kernels cuánticos con ruido (matriz de densidad, depolarizante).
N_JOBS = 1            # >1 paraleliza con joblib por (dataset, semilla, pliegue, kernel).
# -----------------------------------------------------------------------------------

CONFIG_FULL = dict(
    # dataset -> PCA k y submuestra estratificada (None = todas las muestras)
    datasets={
        "breast_cancer": dict(scaler="standard", reducer="pca", k=4, subsample=None),
        "banknote":      dict(scaler="standard", reducer="pca", k=4, subsample=300),
        "haberman":      dict(scaler="standard", reducer="pca", k=3, subsample=None),
    },
    quantum_kernels=["zzfm", "rot2dof"],
    classical_kernels=["rbf", "linear"],
    reps=2,
    modes=["kakavand", "scale_tuned"],
    C_grid=[0.01, 0.1, 1.0, 10.0, 100.0],
    # Extensión propia (Shaydulin & Wild 2022; Flórez et al. 2025), NO de Kakavand:
    s_grid=[0.1, 0.25, 0.5, 1.0, 2.0],
    n_outer=5,
    n_inner=3,
    seeds=[0, 1, 2, 3, 4],    # cambian particiones (externas e internas) y la submuestra
    subsample_seed=None,      # None -> la submuestra usa la semilla de la corrida; int -> fija
    noise=dict(p1=0.001, p2=0.01, shots=1024),
    kernel_cache_dir="results/qsvc/kernel_cache",   # compartido entre corridas; None desactiva
)
CONFIG_QUICK = {**CONFIG_FULL, "seeds": [0]}
CONFIG = CONFIG_QUICK if QUICK else CONFIG_FULL
CONFIG = {**CONFIG, "quick": QUICK, "noise_enabled": NOISE, "n_jobs": N_JOBS}

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
warnings.filterwarnings("ignore", category=DeprecationWarning)
print("ROOT:", ROOT)
print(json.dumps(CONFIG, indent=2))

## 2. Utilidades (semillas, versiones, logging, checkpoint)

In [ ]:
import random

def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)

def package_versions():
    import importlib.metadata as md_
    names = ["numpy", "scipy", "pandas", "scikit-learn", "qiskit", "qiskit-aer",
             "qiskit-machine-learning", "qsvm-benchmark", "ucimlrepo"]
    out = {}
    for n in names:
        try:
            out[n] = md_.version(n)
        except md_.PackageNotFoundError:
            out[n] = None
    return out

def git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return None

def setup_output(output_dir, resume):
    if output_dir is None:
        out = ROOT / "results" / "qsvc" / datetime.now().strftime("%Y%m%d_%H%M%S")
    else:
        out = Path(output_dir)
        if not out.is_absolute():
            out = ROOT / out
    if out.exists() and not resume and any(out.iterdir()):
        raise FileExistsError(f"{out} ya existe; usar RESUME=True o otro OUTPUT_DIR")
    out.mkdir(parents=True, exist_ok=True)
    return out

def get_logger(out_dir):
    log = logging.getLogger("qsvc")
    log.setLevel(logging.INFO)
    log.handlers.clear()
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    for h in (logging.StreamHandler(), logging.FileHandler(out_dir / "run.log", encoding="utf-8")):
        h.setFormatter(fmt)
        log.addHandler(h)
    return log

FIELDS = [
    "dataset", "n_samples", "n_features", "n_qubits", "kernel", "feature_map", "reps", "mode",
    "seed", "outer_fold", "best_C", "best_s", "balanced_accuracy", "accuracy", "n_train", "n_test",
    "n_kernel_evals", "kernel_time_s", "fit_time_s",
    # columnas adicionales (no pedidas en §6.5, útiles para auditar)
    "backend", "inner_cv_score", "kernel_cache_hit", "subsample_seed",
]

def run_key(dataset, kernel, mode, seed, outer_fold):
    return f"{dataset}|{kernel}|{mode}|{int(seed)}|{int(outer_fold)}"

def load_done(csv_path):
    if not csv_path.exists():
        return set()
    df = pd.read_csv(csv_path)
    return {run_key(r.dataset, r.kernel, r["mode"], r.seed, r.outer_fold) for _, r in df.iterrows()}

def append_rows(csv_path, rows):
    # checkpoint: cada corrida se guarda apenas termina
    new = not csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        for row in rows:
            w.writerow({k: row.get(k) for k in FIELDS})

## 3. Datos y preprocesamiento

Los datasets se cargan con `DatasetRegistry.get(name)` de Kakavand (`banknote` y `haberman` se
descargan de UCI, con respaldo en OpenML). `banknote` se submuestrea de forma estratificada a 300
puntos; la semilla de la submuestra es la de la corrida (o `subsample_seed` si se fija) y los
índices se guardan en `subsample_indices.json`.

El preprocesamiento (`build_preprocessor` de Kakavand: `StandardScaler` + PCA) se crea **nuevo en
cada pliegue externo** y se ajusta solo con su entrenamiento. Las etiquetas de prueba no se le
pasan (se entrega un vector de ceros, como en el runner de Kakavand).

In [ ]:
_DATA_CACHE = {}

def load_raw(name):
    if name not in _DATA_CACHE:
        _DATA_CACHE[name] = DatasetRegistry.get(name)
    return _DATA_CACHE[name]

def load_data(name, spec, seed, cfg):
    # devuelve X, y (posiblemente submuestreados), índices de la submuestra y su semilla
    ds = load_raw(name)
    X, y = ds.X, ds.y
    idx = np.arange(len(y))
    sub_seed = None
    if spec.get("subsample"):
        sub_seed = seed if cfg["subsample_seed"] is None else int(cfg["subsample_seed"])
        idx, _ = train_test_split(idx, train_size=spec["subsample"], stratify=y, random_state=sub_seed)
        idx = np.sort(idx)
    return X[idx], y[idx], idx, sub_seed

def preprocess_fold(X_tr, y_tr, X_te, spec, seed):
    # pipeline nuevo por pliegue externo, ajustado solo con X_tr
    pp = build_preprocessor(scaler=spec["scaler"], reducer=spec["reducer"],
                            k_features=spec["k"], random_state=seed)
    res = pp.fit_transform(X_tr, y_tr, X_te, np.zeros(len(X_te)))
    return res.X_train, res.X_test, pp

## 4. Kernels

- **Cuánticos** (`zzfm`, `rot2dof`, `reps = 2`): `compute_statevector_kernel` de Kakavand,
  `K(x, x') = |⟨ψ(x)|ψ(x')⟩|²` exacto. Con `NOISE = True` se agregan `zzfm_noisy` / `rot2dof_noisy`
  con `compute_noisy_density_matrix_kernel` (`K = Tr[ρ(x)ρ(x')]`, depolarizante 1q = 0.001,
  2q = 0.01, modelo de ruido de Kakavand).
- **Clásicos:** `rbf` con `γ = 'scale'` calculado **sobre el entrenamiento del pliegue a escala 1**
  (`gamma_scale` de Kakavand) y `linear` (`LinearKernel` de Kakavand).
- **Escala `s`** (extensión propia, Shaydulin & Wild / Flórez): `x → s·x` antes del kernel. En el
  RBF, como `γ` se fija con los datos a escala 1, equivale a multiplicar `γ` por `s²`. En el lineal
  `s` solo reescala `C` (`K → s²K`), así que **no se aplica**: el lineal usa siempre `s = 1`.
  Nota: el `RBFKernel` de Kakavand calcula `γ` con el primer argumento, por lo que en
  `K(X_test, X_train)` usaría la varianza del test; aquí se fija con el entrenamiento.
- Cada matriz se calcula una vez por pliegue y por `s` y se reutiliza para todos los `C` (y para
  ambos modos: `kakavand` usa la de `s = 1`). Las matrices cuánticas se cachean en disco
  (`KernelCache` de Kakavand, clave `xxhash` de datos + configuración); las clásicas se recalculan
  (son instantáneas).
- `n_kernel_evals` = entradas de Gram calculadas: `n_train(n_train+1)/2 + n_test·n_train` por cada
  valor de `s` usado por el modo (equivale al número de circuitos de fidelidad que necesitaría la
  estimación compute–uncompute en hardware; en statevector se simulan `n_train + n_test` estados).

In [ ]:
QUANTUM_FMS = {"zzfm", "rot2dof"}

def kernel_info(kernel, n_features, cfg):
    # metadatos fijos de un kernel: mapa de características, qubits, backend
    fm = kernel.removesuffix("_noisy")
    if fm in QUANTUM_FMS:
        n_q = build_feature_map(fm, n_features, reps=cfg["reps"]).n_qubits
        return dict(feature_map=fm, n_qubits=n_q, reps=cfg["reps"],
                    backend="noisy_dm" if kernel.endswith("_noisy") else "statevector")
    return dict(feature_map=None, n_qubits=None, reps=None, backend="classical")

def kernels_for(cfg):
    ks = list(cfg["quantum_kernels"])
    if cfg["noise_enabled"]:
        ks += [f"{k}_noisy" for k in cfg["quantum_kernels"]]
    return ks + list(cfg["classical_kernels"])

def s_values(kernel, mode, cfg):
    if mode == "kakavand" or kernel == "linear":
        return [1.0]
    if mode == "scale_tuned":
        return list(cfg["s_grid"])
    raise ValueError(mode)

def _quantum_matrices(kernel, Xs_tr, Xs_te, cfg, seed):
    fm = kernel.removesuffix("_noisy")
    nf = Xs_tr.shape[1]
    if kernel.endswith("_noisy"):
        nz = cfg["noise"]
        backend = create_noisy_backend(NoiseConfig(type="depolarizing", depol_1q=nz["p1"], depol_2q=nz["p2"]),
                                       seed=seed)
        f = lambda A, B=None: compute_noisy_density_matrix_kernel(
            fm, nf, A, B, backend=backend, shots=nz["shots"], seed=seed, reps=cfg["reps"], show_progress=False)
    else:
        f = lambda A, B=None: compute_statevector_kernel(fm, nf, A, B, reps=cfg["reps"], show_progress=False)
    return f(Xs_tr), f(Xs_te, Xs_tr)

def kernel_matrices(kernel, X_tr, X_te, s, cfg, seed, cache=None):
    """K(X_tr, X_tr) y K(X_te, X_tr) para la escala s. Devuelve (K_tr, K_te, compute_time_s, cache_hit)."""
    Xs_tr, Xs_te = s * X_tr, s * X_te
    if kernel == "linear":
        t0 = time.time()
        lk = LinearKernel()
        return lk.compute(X_tr), lk.compute(X_te, X_tr), time.time() - t0, False
    if kernel == "rbf":
        t0 = time.time()
        g = gamma_scale(X_tr)              # 'scale' sobre el entrenamiento a escala 1 -> gamma·s²
        return rbf_kernel(Xs_tr, gamma=g), rbf_kernel(Xs_te, Xs_tr, gamma=g), time.time() - t0, False

    key_cfg = dict(kernel=kernel, reps=cfg["reps"])
    if kernel.endswith("_noisy"):
        key_cfg.update(noise=cfg["noise"], seed=seed)
    key = compute_kernel_hash(np.ascontiguousarray(Xs_tr), np.ascontiguousarray(Xs_te), key_cfg)
    if cache is not None:
        hit = cache.load(key)
        if hit is not None:
            K_tr, K_te, meta = hit
            return K_tr, K_te, float(meta.get("compute_time_s", np.nan)), True
    t0 = time.time()
    K_tr, K_te = _quantum_matrices(kernel, Xs_tr, Xs_te, cfg, seed)
    dt = time.time() - t0
    if cache is not None:
        cache.save(key, K_tr, K_te, metadata=dict(compute_time_s=dt, **key_cfg))
    return K_tr, K_te, dt, False

def n_gram_entries(n_tr, n_te):
    return n_tr * (n_tr + 1) // 2 + n_te * n_tr

## 5. CV anidada (bucle propio)

Para cada semilla y pliegue externo (`StratifiedKFold(5, shuffle=True, random_state=seed)`):
1. Preprocesamiento ajustado con el entrenamiento externo.
2. Para cada kernel y cada `s` necesario: matrices `K_tr` y `K_te`.
3. CV interna (`StratifiedKFold(3, shuffle=True, random_state=seed)`, como Kakavand) sobre
   submatrices de `K_tr`; se elige el par `(C, s)` con mayor exactitud balanceada media.
   Desempate: gana el primer candidato en el orden `s = 1` primero, luego el resto de `s_grid`;
   dentro de cada `s`, `C` ascendente (en modo `kakavand` coincide con `_inner_cv_grid_search`).
4. Se reentrena `SVC(kernel="precomputed")` con el par elegido sobre todo el entrenamiento externo
   y se evalúa en el test externo.

In [ ]:
def inner_search(K_by_s, y_tr, inner_splits, C_grid):
    """Búsqueda en rejilla (s, C) sobre la CV interna. K_by_s: {s: K_tr} ya ordenado para el desempate."""
    best = (None, None, -np.inf)
    for s, K in K_by_s.items():
        for C in C_grid:
            scores = []
            for a, b in inner_splits:
                clf = SVC(kernel="precomputed", C=C).fit(K[np.ix_(a, a)], y_tr[a])
                scores.append(balanced_accuracy_score(y_tr[b], clf.predict(K[np.ix_(b, a)])))
            m = float(np.mean(scores))
            if m > best[2]:
                best = (C, s, m)
    return best

def evaluate_fold(X_tr, y_tr, X_te, y_te, kernel, modes, seed, cfg, cache=None):
    """Evalúa un kernel en un pliegue externo (datos ya preprocesados). Devuelve {mode: dict}."""
    inner = list(StratifiedKFold(cfg["n_inner"], shuffle=True, random_state=seed)
                 .split(np.zeros(len(y_tr)), y_tr))
    order = [1.0] + [s for s in cfg["s_grid"] if s != 1.0]          # orden de desempate
    needed = [s for s in order if any(s in s_values(kernel, m, cfg) for m in modes)]
    mats = {}
    for s in needed:
        mats[s] = kernel_matrices(kernel, X_tr, X_te, s, cfg, seed, cache)
    out = {}
    for mode in modes:
        t0 = time.time()
        svals = [s for s in needed if s in s_values(kernel, mode, cfg)]   # conserva el orden de desempate
        C, s, score = inner_search({s_: mats[s_][0] for s_ in svals}, y_tr, inner, cfg["C_grid"])
        K_tr, K_te = mats[s][0], mats[s][1]
        clf = SVC(kernel="precomputed", C=C).fit(K_tr, y_tr)
        y_pred = clf.predict(K_te)
        out[mode] = dict(
            best_C=C, best_s=s, inner_cv_score=score,
            balanced_accuracy=float(balanced_accuracy_score(y_te, y_pred)),
            accuracy=float(accuracy_score(y_te, y_pred)),
            n_kernel_evals=len(svals) * n_gram_entries(len(y_tr), len(y_te)),
            kernel_time_s=float(sum(mats[s_][2] for s_ in svals)),
            kernel_cache_hit=bool(all(mats[s_][3] for s_ in svals)),
            fit_time_s=time.time() - t0,
        )
    return out

def outer_splits(y, seed, cfg):
    return list(StratifiedKFold(cfg["n_outer"], shuffle=True, random_state=seed).split(np.zeros(len(y)), y))

def run_unit(dataset, seed, fold, kernel, modes, cfg, cache_dir):
    """Una unidad de trabajo (dataset, semilla, pliegue, kernel): filas para todos los modos pedidos."""
    set_seeds(seed)
    spec = cfg["datasets"][dataset]
    X, y, _, sub_seed = load_data(dataset, spec, seed, cfg)
    tr, te = outer_splits(y, seed, cfg)[fold]
    X_tr, X_te, _ = preprocess_fold(X[tr], y[tr], X[te], spec, seed)
    cache = KernelCache(cache_dir) if cache_dir else None
    res = evaluate_fold(X_tr, y[tr], X_te, y[te], kernel, modes, seed, cfg, cache)
    info = kernel_info(kernel, X_tr.shape[1], cfg)
    base = dict(dataset=dataset, n_samples=len(y), n_features=X_tr.shape[1], kernel=kernel, seed=seed,
                outer_fold=fold, n_train=len(tr), n_test=len(te), subsample_seed=sub_seed, **info)
    return [{**base, "mode": m, **r} for m, r in res.items()]

## 6. Pruebas mínimas (`assert`)

1. El preprocesamiento no ve datos de prueba del pliegue externo.
2. Con `s = 1` y `mode = kakavand`, el RBF reproduce el smoke test de Kakavand en `breast_cancer`
   (PCA k = 4, CV 2×2, `C ∈ {1, 10}`, semilla 42): exactitud balanceada ≈ 0,945 (tolerancia 0,01).
   Su runner da 0,9450; el bucle propio da algo menos porque fija `γ` con el entrenamiento, mientras
   que el `RBFKernel` de Kakavand lo recalcula con el test en `K(X_test, X_train)`. Con el `γ` de
   Kakavand el bucle propio coincide exactamente con su runner (se comprueba).
3. Las matrices kernel son simétricas, con diagonal 1 (cuánticos ideales y RBF) y semidefinidas
   positivas salvo error numérico.
4. Equivalencia con Qiskit ML (`FidelityQuantumKernel` / `QSVC`). Si `qiskit-machine-learning`
   0.8.2 no es compatible con `qiskit` 2.3, la prueba se **omite** y se documenta; en su lugar se
   comprueba el mismo kernel con el circuito compute–uncompute `U†(x')U(x)` simulado en
   statevector (es la definición que usa `FidelityQuantumKernel`).
5. (Adicional) En modo `kakavand`, el bucle propio reproduce pliegue a pliegue (`C` elegido y
   exactitud balanceada) el runner leakage-free de Kakavand con `zzfm` en `breast_cancer`.

In [ ]:
SPEC_BC = CONFIG["datasets"]["breast_cancer"]
Xb, yb, _, _ = load_data("breast_cancer", SPEC_BC, 0, CONFIG)
tr0, te0 = outer_splits(yb, 0, CONFIG)[0]

# Prueba 1: sin fuga en el preprocesamiento
Xp_tr, Xp_te, pp0 = preprocess_fold(Xb[tr0], yb[tr0], Xb[te0], SPEC_BC, 0)
assert np.allclose(pp0._scaler.mean_, Xb[tr0].mean(axis=0)), "el escalador usó datos fuera del entrenamiento"
X_te_mod = Xb[te0] * 1e3 + 7.0                       # test alterado: el train transformado no debe cambiar
Xp_tr2, _, pp1 = preprocess_fold(Xb[tr0], yb[tr0], X_te_mod, SPEC_BC, 0)
assert np.array_equal(Xp_tr, Xp_tr2), "la transformación del entrenamiento depende del test"
assert np.allclose(pp0._reducer.components_, pp1._reducer.components_)
assert np.allclose(Xp_te, pp0._reducer.transform(pp0._scaler.transform(Xb[te0]))), "test no transformado con el ajuste de train"
assert len(np.intersect1d(tr0, te0)) == 0
print("Prueba 1 OK")

# Prueba 2: smoke test de Kakavand con el RBF (s = 1, modo kakavand)
from qsvm_benchmark.experiments.nested_cv import NestedCVConfig
from qsvm_benchmark.experiments.runner import BenchmarkRunner, ExperimentConfig
smoke_cfg = {**CONFIG, "n_outer": 2, "n_inner": 2, "C_grid": [1.0, 10.0]}
bas = []
for tr, te in outer_splits(yb, 42, smoke_cfg):
    a_tr, a_te, _ = preprocess_fold(Xb[tr], yb[tr], Xb[te], SPEC_BC, 42)
    bas.append(evaluate_fold(a_tr, yb[tr], a_te, yb[te], "rbf", ["kakavand"], 42, smoke_cfg)["kakavand"]["balanced_accuracy"])
ba_ours = float(np.mean(bas))
with tempfile.TemporaryDirectory() as tmp:
    ref = BenchmarkRunner(output_dir=tmp, enable_cache=False).run_experiment(
        ExperimentConfig(dataset="breast_cancer", reducer="pca", k_features=4, kernel_type="rbf", seed=42,
                         cv_config=NestedCVConfig(n_outer_folds=2, n_inner_folds=2, C_grid=[1.0, 10.0], random_state=42)),
        leakage_free=True)
ba_ref = ref.cv_result.mean_metrics["balanced_accuracy"]
# La diferencia esperada viene solo de gamma: el RBFKernel de Kakavand calcula gamma='scale' con el
# primer argumento, así que en K(X_test, X_train) usa la varianza del test. Con ese mismo cálculo
# (y el resto del bucle propio) el resultado debe coincidir exactamente con su runner.
from qsvm_benchmark.kernels import RBFKernel
bas_k = []
for tr, te in outer_splits(yb, 42, smoke_cfg):
    a_tr, a_te, _ = preprocess_fold(Xb[tr], yb[tr], Xb[te], SPEC_BC, 42)
    K_tr_k, K_te_k = RBFKernel().compute(a_tr), RBFKernel().compute(a_te, a_tr)
    inn = list(StratifiedKFold(2, shuffle=True, random_state=42).split(np.zeros(len(tr)), yb[tr]))
    C_k, _, _ = inner_search({1.0: K_tr_k}, yb[tr], inn, smoke_cfg["C_grid"])
    bas_k.append(balanced_accuracy_score(yb[te], SVC(kernel="precomputed", C=C_k).fit(K_tr_k, yb[tr]).predict(K_te_k)))
print(f"Prueba 2: BA propia = {ba_ours:.4f} | runner de Kakavand = {ba_ref:.4f} | "
      f"propia con gamma de Kakavand = {np.mean(bas_k):.4f} | esperado ≈ 0.945")
assert abs(ba_ours - 0.945) < 0.01, "el RBF no reproduce el smoke test de Kakavand"
assert abs(np.mean(bas_k) - ba_ref) < 1e-12, "el bucle propio no reproduce el runner de Kakavand"

# Prueba 3: propiedades de las matrices kernel (s = 1 y s = 0.25, pliegue 0 de breast_cancer)
for kern in ["zzfm", "rot2dof", "rbf", "linear"]:
    for s in (1.0, 0.25):
        K, K_te, _, _ = kernel_matrices(kern, Xp_tr, Xp_te, s, CONFIG, 0)
        assert K.shape == (len(tr0), len(tr0)) and K_te.shape == (len(te0), len(tr0))
        assert np.allclose(K, K.T, atol=1e-10), f"{kern}: no simétrica"
        if kern != "linear":
            assert np.allclose(np.diag(K), 1.0, atol=1e-10), f"{kern}: diagonal != 1"
            assert K.min() >= -1e-12 and K.max() <= 1 + 1e-10
        lam_min = np.linalg.eigvalsh(K).min()
        assert lam_min > -1e-8 * np.abs(K).max() * len(K), f"{kern}: no PSD (λ_min={lam_min:.2e})"
        print(f"Prueba 3: {kern:8s} s={s:<5} λ_min={lam_min:+.2e} OK")

# Prueba 4: equivalencia con Qiskit ML
try:
    from qiskit_machine_learning.kernels import FidelityQuantumKernel
    from qiskit_machine_learning.algorithms import QSVC
    QML_OK = True
except ImportError as e:
    QML_OK = False
    print(f"Prueba 4 OMITIDA (skip): qiskit-machine-learning no es compatible con qiskit "
          f"{package_versions()['qiskit']}: {e}")

sub = np.arange(60)                                   # subconjunto del pliegue para que sea rápida
for fm_name in ["zzfm", "rot2dof"]:
    fm = build_feature_map(fm_name, Xp_tr.shape[1], reps=CONFIG["reps"])
    K_kak = compute_statevector_kernel(fm_name, Xp_tr.shape[1], Xp_tr[sub], reps=CONFIG["reps"], show_progress=False)
    if QML_OK:
        K_qml = FidelityQuantumKernel(feature_map=fm.circuit).evaluate(Xp_tr[sub])
        assert np.abs(K_qml - K_kak).max() < 1e-8
        svc = SVC(kernel="precomputed", C=1.0).fit(K_kak, yb[tr0][sub])
        qsvc = QSVC(quantum_kernel=FidelityQuantumKernel(feature_map=fm.circuit), C=1.0).fit(Xp_tr[sub], yb[tr0][sub])
        K_te_kak = compute_statevector_kernel(fm_name, Xp_tr.shape[1], Xp_te, Xp_tr[sub], reps=CONFIG["reps"], show_progress=False)
        assert np.array_equal(svc.predict(K_te_kak), qsvc.predict(Xp_te))
        print(f"Prueba 4 ({fm_name}): Qiskit ML coincide")
    # Comprobación alternativa: circuito compute-uncompute U(x) seguido de U†(x'), amplitud de |0…0⟩
    params = list(fm.parameters.values())
    bind = lambda x: fm.circuit.assign_parameters({p: float(v) for p, v in zip(params, x)})
    K_cu = np.empty((10, 10))
    for i in range(10):
        for j in range(10):
            qc = bind(Xp_tr[sub[i]]).compose(bind(Xp_tr[sub[j]]).inverse())
            K_cu[i, j] = abs(Statevector(qc).data[0]) ** 2
    d = np.abs(K_cu - K_kak[:10, :10]).max()
    print(f"Prueba 4 alternativa ({fm_name}): max|ΔK| compute-uncompute vs Kakavand = {d:.2e}")
    assert d < 1e-8

# Prueba 5 (adicional): con un kernel cuántico, el modo kakavand del bucle propio reproduce pliegue a
# pliegue el runner leakage-free de Kakavand (breast_cancer, zzfm, semilla 0, 5×3, C completo).
# Se silencian sus barras de progreso parcheando tqdm en tiempo de ejecución (su código no se toca).
from functools import partial
from unittest import mock
import qsvm_benchmark.kernels.quantum as _kq
with tempfile.TemporaryDirectory() as tmp, mock.patch.object(_kq, "tqdm", partial(tqdm, disable=True)):
    ref5 = BenchmarkRunner(output_dir=tmp, enable_cache=False).run_experiment(
        ExperimentConfig(dataset="breast_cancer", reducer="pca", k_features=4, kernel_type="zzfm", seed=0,
                         reps=CONFIG["reps"], cv_config=NestedCVConfig(n_outer_folds=CONFIG["n_outer"],
                         n_inner_folds=CONFIG["n_inner"], C_grid=CONFIG["C_grid"], random_state=0)),
        leakage_free=True)
ours5 = []
for tr, te in outer_splits(yb, 0, CONFIG):
    a_tr, a_te, _ = preprocess_fold(Xb[tr], yb[tr], Xb[te], SPEC_BC, 0)
    r5 = evaluate_fold(a_tr, yb[tr], a_te, yb[te], "zzfm", ["kakavand"], 0, CONFIG)["kakavand"]
    ours5.append((r5["best_C"], r5["balanced_accuracy"]))
ref5 = [(f.best_C, f.metrics["balanced_accuracy"]) for f in ref5.cv_result.fold_results]
print("Prueba 5: (C, BA) por pliegue, propio vs Kakavand:", [(c, round(b, 4)) for c, b in ours5])
assert all(c1 == c2 and abs(b1 - b2) < 1e-12 for (c1, b1), (c2, b2) in zip(ours5, ref5))
print("Pruebas OK")

## 7. Barrido (con checkpoint por corrida)

Unidad de trabajo: `(dataset, semilla, pliegue externo, kernel)`; cada una escribe una fila por
modo apenas termina. Con `RESUME = True` se saltan las filas ya presentes en `runs.csv`. Las
matrices cuánticas quedan en `kernel_cache_dir`, de modo que una corrida repetida o retomada no
las recalcula.

In [ ]:
out_dir = setup_output(OUTPUT_DIR, RESUME)
log = get_logger(out_dir)
csv_path = out_dir / "runs.csv"
cache_dir = None if CONFIG["kernel_cache_dir"] is None else str(ROOT / CONFIG["kernel_cache_dir"])

# metadatos: config resuelta, versiones, datasets e índices de submuestreo
ds_meta, sub_idx = {}, {}
for name, spec in CONFIG["datasets"].items():
    raw = load_raw(name)
    ds_meta[name] = dict(n_samples_raw=raw.n_samples, n_features_raw=raw.n_features,
                         source=raw.metadata.get("source"), class_counts=np.bincount(raw.y).tolist())
    if spec.get("subsample"):
        sub_idx[name] = {}
        for seed in CONFIG["seeds"]:
            _, _, idx, ss = load_data(name, spec, seed, CONFIG)
            sub_idx[name][str(seed)] = dict(subsample_seed=ss, indices=idx.tolist())
with open(out_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(dict(config=CONFIG, versions=package_versions(), python=platform.python_version(),
                   platform=platform.platform(), git_commit=git_commit(),
                   date=datetime.now().isoformat(timespec="seconds"), datasets=ds_meta,
                   notes=dict(scale_s="El ajuste de s (modo scale_tuned) es una extensión propia motivada por "
                                      "Shaydulin & Wild 2022 y Flórez et al. 2025; no es parte del protocolo de Kakavand.",
                              linear_s="El kernel lineal usa siempre s=1 (s solo reescalaría C).",
                              rbf_gamma="gamma='scale' calculado con el entrenamiento del pliegue a escala 1.")),
              f, indent=2)
with open(out_dir / "subsample_indices.json", "w", encoding="utf-8") as f:
    json.dump(sub_idx, f)

done = load_done(csv_path) if RESUME else set()
units = []
for dataset, seed, fold, kernel in itertools.product(CONFIG["datasets"], CONFIG["seeds"],
                                                     range(CONFIG["n_outer"]), kernels_for(CONFIG)):
    modes = [m for m in CONFIG["modes"] if run_key(dataset, kernel, m, seed, fold) not in done]
    if modes:
        units.append((dataset, seed, fold, kernel, modes))
log.info("Salida: %s | %d unidades pendientes (%d filas ya hechas)", out_dir, len(units), len(done))

def _log_rows(rows):
    for r in rows:
        log.info("%s BA=%.4f C=%g s=%g k_t=%.2fs fit_t=%.2fs%s", run_key(r["dataset"], r["kernel"], r["mode"],
                 r["seed"], r["outer_fold"]), r["balanced_accuracy"], r["best_C"], r["best_s"],
                 r["kernel_time_s"], r["fit_time_s"], " (cache)" if r["kernel_cache_hit"] else "")

if N_JOBS == 1:
    for u in tqdm(units):
        try:
            rows = run_unit(*u, CONFIG, cache_dir)
        except Exception:
            log.exception("Falló %s", u)
            continue
        append_rows(csv_path, rows)
        _log_rows(rows)
else:
    # Paralelo: los resultados llegan en cualquier orden y se guardan apenas llegan.
    # En Windows las funciones definidas en celdas pueden fallar al serializarse; usar N_JOBS = 1.
    from joblib import Parallel, delayed
    def _safe(u):
        try:
            return u, run_unit(*u, CONFIG, cache_dir), None
        except Exception as e:
            return u, None, repr(e)
    for u, rows, err in tqdm(Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(delayed(_safe)(u) for u in units),
                             total=len(units)):
        if err:
            log.error("Falló %s: %s", u, err)
            continue
        append_rows(csv_path, rows)
        _log_rows(rows)
log.info("Terminado")

## 8. Vista previa de los resultados crudos

In [ ]:
df = pd.read_csv(csv_path)
print(len(df), "filas en", csv_path)
df[["dataset", "kernel", "mode", "seed", "outer_fold", "n_qubits", "best_C", "best_s",
    "balanced_accuracy", "accuracy", "n_train", "n_test", "kernel_time_s", "fit_time_s"]].head(10)